<a href="https://colab.research.google.com/github/nishilkanjiya640/Employee-wellness-management/blob/main/Copy_of_login_page.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |


neondb_owner

In [1]:
!pip install -q streamlit==1.38.0 psycopg2-binary==2.9.9 PyJWT==2.9.0 bcrypt==4.2.0 \
    python-dotenv==1.0.1 email-validator==2.2.0 pyngrok==7.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.3.0 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.
google-adk 2.3.0 requires watchdog<7,>=6, but you have watchdog 4.0.2 which is incompatible.


In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
"""
db.py
PostgreSQL connection handling + schema initialization.
"""

import os
import psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "dbname": os.getenv("DB_NAME", "auth_app"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", ""),
    "sslmode": "require",
}


@contextmanager
def get_connection():
    """Yield a PostgreSQL connection, always closed on exit."""
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        yield conn
    finally:
        conn.close()


@contextmanager
def get_cursor(commit: bool = False):
    """Yield a RealDictCursor. Commits if commit=True and no exception occurred."""
    with get_connection() as conn:
        cur = conn.cursor(cursor_factory=RealDictCursor)
        try:
            yield cur
            if commit:
                conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cur.close()


def init_db():
    """Create tables if they don't exist. Safe to call on every app startup."""
    with get_cursor(commit=True) as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id SERIAL PRIMARY KEY,
                username VARCHAR(50) UNIQUE NOT NULL,
                email VARCHAR(255) UNIQUE NOT NULL,
                password_hash VARCHAR(255) NOT NULL,
                is_verified BOOLEAN NOT NULL DEFAULT FALSE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id SERIAL PRIMARY KEY,
                email VARCHAR(255) NOT NULL,
                code VARCHAR(6) NOT NULL,
                purpose VARCHAR(20) NOT NULL,
                expires_at TIMESTAMP NOT NULL,
                used BOOLEAN NOT NULL DEFAULT FALSE,
                created_at TIMESTAMP NOT NULL DEFAULT NOW()
            );
        """)

        cur.execute("""
            CREATE INDEX IF NOT EXISTS idx_otp_email_purpose
            ON otp_codes (email, purpose);
        """)
    print("Database initialized (tables ensured).")


if __name__ == "__main__":
    init_db()


Writing db.py


In [4]:
%%writefile auth.py
"""
auth.py
Password hashing, JWT issuing/verification, and user data-access functions.
"""

import os
import jwt
import bcrypt
import random
import string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

from db import get_cursor

load_dotenv()

JWT_SECRET = os.getenv("JWT_SECRET")
JWT_ALGORITHM = os.getenv("JWT_ALGORITHM", "HS256")
JWT_EXPIRY_MINUTES = int(os.getenv("JWT_EXPIRY_MINUTES", "60"))
OTP_EXPIRY_MINUTES = int(os.getenv("OTP_EXPIRY_MINUTES", "10"))


def hash_password(plain_password: str) -> str:
    return bcrypt.hashpw(plain_password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(plain_password: str, password_hash: str) -> bool:
    return bcrypt.checkpw(plain_password.encode("utf-8"), password_hash.encode("utf-8"))


def create_jwt(user_id: int, username: str, email: str) -> str:
    payload = {
        "user_id": user_id,
        "username": username,
        "email": email,
        "exp": datetime.now(timezone.utc) + timedelta(minutes=JWT_EXPIRY_MINUTES),
        "iat": datetime.now(timezone.utc),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)


def decode_jwt(token: str):
    """Returns the payload dict if valid, else None."""
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALGORITHM])
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None


def get_user_by_email(email: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email = %s;", (email,))
        return cur.fetchone()


def get_user_by_username(username: str):
    with get_cursor() as cur:
        cur.execute("SELECT * FROM users WHERE username = %s;", (username,))
        return cur.fetchone()


def create_user(username: str, email: str, password: str):
    password_hash = hash_password(password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            INSERT INTO users (username, email, password_hash, is_verified)
            VALUES (%s, %s, %s, FALSE)
            RETURNING id, username, email;
            """,
            (username, email, password_hash),
        )
        return cur.fetchone()


def mark_user_verified(email: str):
    with get_cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified = TRUE WHERE email = %s;", (email,))


def update_user_password(email: str, new_password: str):
    password_hash = hash_password(new_password)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET password_hash = %s WHERE email = %s;",
            (password_hash, email),
        )


def generate_otp() -> str:
    return "".join(random.choices(string.digits, k=6))


def store_otp(email: str, code: str, purpose: str):
    """purpose: 'signup' or 'password_reset'"""
    expires_at = datetime.now(timezone.utc) + timedelta(minutes=OTP_EXPIRY_MINUTES)
    with get_cursor(commit=True) as cur:
        cur.execute(
            "UPDATE otp_codes SET used = TRUE WHERE email = %s AND purpose = %s AND used = FALSE;",
            (email, purpose),
        )
        cur.execute(
            """
            INSERT INTO otp_codes (email, code, purpose, expires_at)
            VALUES (%s, %s, %s, %s);
            """,
            (email, code, purpose, expires_at),
        )


def verify_otp(email: str, code: str, purpose: str) -> bool:
    with get_cursor(commit=True) as cur:
        cur.execute(
            """
            SELECT * FROM otp_codes
            WHERE email = %s AND purpose = %s AND used = FALSE
            ORDER BY created_at DESC LIMIT 1;
            """,
            (email, purpose),
        )
        row = cur.fetchone()
        if not row:
            return False

        expires_at = row["expires_at"]
        now = datetime.now(expires_at.tzinfo) if expires_at.tzinfo else datetime.now()

        if now > expires_at:
            return False
        if row["code"] != code:
            return False

        cur.execute("UPDATE otp_codes SET used = TRUE WHERE id = %s;", (row["id"],))
        return True


Writing auth.py


In [5]:
%%writefile email_utils.py
"""
email_utils.py
Sends OTP emails using Gmail SMTP (smtp.gmail.com, port 587, STARTTLS).
"""

import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from dotenv import load_dotenv

load_dotenv()

SMTP_HOST = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT = int(os.getenv("SMTP_PORT", "587"))
SMTP_EMAIL = os.getenv("SMTP_EMAIL")
SMTP_APP_PASSWORD = os.getenv("SMTP_APP_PASSWORD")


def send_email(to_email: str, subject: str, body: str):
    if not SMTP_EMAIL or not SMTP_APP_PASSWORD:
        return False, "SMTP credentials are not configured in .env"

    msg = MIMEMultipart()
    msg["From"] = SMTP_EMAIL
    msg["To"] = to_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=15) as server:
            server.ehlo()
            server.starttls()
            server.ehlo()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.sendmail(SMTP_EMAIL, to_email, msg.as_string())
        return True, "Email sent."
    except smtplib.SMTPAuthenticationError:
        return False, "SMTP auth failed. Check SMTP_EMAIL / SMTP_APP_PASSWORD (use a Gmail App Password)."
    except Exception as e:
        return False, f"Failed to send email: {e}"


def send_otp_email(to_email: str, otp_code: str, purpose: str):
    if purpose == "signup":
        subject = "Your Verification Code"
        body = (
            f"Welcome!\n\nYour verification code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    else:
        subject = "Your Password Reset Code"
        body = (
            f"We received a request to reset your password.\n\n"
            f"Your reset code is: {otp_code}\n\n"
            f"This code expires in a few minutes. If you did not request this, "
            f"you can safely ignore this email."
        )
    return send_email(to_email, subject, body)


Writing email_utils.py


In [6]:
%%writefile app.py
"""
app.py
Streamlit login/signup/password-reset app backed by PostgreSQL,
using JWT session tokens and Gmail SMTP for OTP delivery.
"""

import re
import streamlit as st
from email_validator import validate_email, EmailNotValidError

from db import init_db
from auth import (
    create_jwt,
    decode_jwt,
    get_user_by_email,
    get_user_by_username,
    create_user,
    mark_user_verified,
    update_user_password,
    verify_password,
    generate_otp,
    store_otp,
    verify_otp,
)
from email_utils import send_otp_email

st.set_page_config(page_title="Secure Login", page_icon="🔐", layout="centered")


@st.cache_resource
def _init_db_once():
    init_db()
    return True

_init_db_once()

defaults = {
    "page": "login",
    "jwt_token": None,
    "pending_email": None,
    "pending_username": None,
}
for key, val in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = val


def goto(page: str):
    st.session_state.page = page
    st.rerun()


def is_valid_password(pw: str):
    if len(pw) < 8:
        return False, "Password must be at least 8 characters."
    if not re.search(r"[A-Za-z]", pw) or not re.search(r"[0-9]", pw):
        return False, "Password must contain both letters and numbers."
    return True, ""


if st.session_state.jwt_token:
    payload = decode_jwt(st.session_state.jwt_token)
    if payload:
        st.title("🔐 Welcome")
        username_ = payload["username"]
        email_ = payload["email"]
        exp_ = payload["exp"]
        st.success(f"Logged in as **{username_}** ({email_})")
        st.caption(f"Token expires (unix): {exp_}")

        with st.expander("Decoded JWT payload"):
            st.json(payload)

        if st.button("Log out"):
            st.session_state.jwt_token = None
            goto("login")
        st.stop()
    else:
        st.session_state.jwt_token = None
        st.warning("Your session expired. Please log in again.")

st.title("🔐 Secure Access")

if st.session_state.page == "login":
    st.subheader("Log in")

    with st.form("login_form"):
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")
        submitted = st.form_submit_button("Log in", use_container_width=True)

    if submitted:
        if not email or not password:
            st.error("Please fill in both fields.")
        else:
            user = get_user_by_email(email.strip().lower())
            if not user or not verify_password(password, user["password_hash"]):
                st.error("Invalid email or password.")
            elif not user["is_verified"]:
                st.warning("This account is not verified yet. Please verify your email.")
                st.session_state.pending_email = user["email"]
                st.session_state.pending_username = user["username"]
                if st.button("Resend verification code"):
                    code_ = generate_otp()
                    store_otp(user["email"], code_, "signup")
                    ok, msg = send_otp_email(user["email"], code_, "signup")
                    if ok:
                        st.info("Verification code sent.")
                        goto("verify_signup")
                    else:
                        st.error(msg)
            else:
                token = create_jwt(user["id"], user["username"], user["email"])
                st.session_state.jwt_token = token
                st.rerun()

    col1, col2 = st.columns(2)
    with col1:
        if st.button("Create an account", use_container_width=True):
            goto("signup")
    with col2:
        if st.button("Forgot password?", use_container_width=True):
            goto("forgot")

elif st.session_state.page == "signup":
    st.subheader("Create an account")

    with st.form("signup_form"):
        username = st.text_input("Username")
        email = st.text_input("Email")
        password = st.text_input("Password", type="password")
        confirm = st.text_input("Confirm password", type="password")
        submitted = st.form_submit_button("Sign up", use_container_width=True)

    if submitted:
        errors = []
        if not username or len(username) < 3:
            errors.append("Username must be at least 3 characters.")

        try:
            valid = validate_email(email, check_deliverability=False)
            email = valid.normalized.lower()
        except EmailNotValidError as e:
            errors.append(f"Invalid email: {e}")

        ok_pw, pw_msg = is_valid_password(password)
        if not ok_pw:
            errors.append(pw_msg)
        if password != confirm:
            errors.append("Passwords do not match.")

        if not errors:
            if get_user_by_username(username):
                errors.append("Username already taken.")
            if get_user_by_email(email):
                errors.append("An account with this email already exists.")

        if errors:
            for e in errors:
                st.error(e)
        else:
            create_user(username, email, password)
            code_ = generate_otp()
            store_otp(email, code_, "signup")
            ok, msg = send_otp_email(email, code_, "signup")
            if ok:
                st.session_state.pending_email = email
                st.session_state.pending_username = username
                st.success("Account created. A verification code was sent to your email.")
                goto("verify_signup")
            else:
                st.error(f"Account created, but the verification email failed to send: {msg}")

    if st.button("← Back to login"):
        goto("login")

elif st.session_state.page == "verify_signup":
    st.subheader("Verify your email")
    email = st.session_state.pending_email
    if not email:
        st.info("No pending verification. Please sign up first.")
        if st.button("Go to signup"):
            goto("signup")
    else:
        st.write(f"Enter the 6-digit code sent to **{email}**.")

        with st.form("verify_form"):
            code_ = st.text_input("Verification code", max_chars=6)
            submitted = st.form_submit_button("Verify", use_container_width=True)

        if submitted:
            if verify_otp(email, code_.strip(), "signup"):
                mark_user_verified(email)
                st.success("Email verified! You can now log in.")
                st.session_state.pending_email = None
                goto("login")
            else:
                st.error("Invalid or expired code.")

        if st.button("Resend code"):
            new_code = generate_otp()
            store_otp(email, new_code, "signup")
            ok, msg = send_otp_email(email, new_code, "signup")
            st.info("A new code was sent.") if ok else st.error(msg)

        if st.button("← Back to login"):
            goto("login")

elif st.session_state.page == "forgot":
    st.subheader("Reset your password")

    with st.form("forgot_form"):
        email = st.text_input("Enter your account email")
        submitted = st.form_submit_button("Send reset code", use_container_width=True)

    if submitted:
        email = email.strip().lower()
        user = get_user_by_email(email)
        if not user:
            st.info("If that email is registered, a reset code has been sent.")
        else:
            code_ = generate_otp()
            store_otp(email, code_, "password_reset")
            ok, msg = send_otp_email(email, code_, "password_reset")
            if ok:
                st.session_state.pending_email = email
                st.success("Reset code sent. Check your email.")
                goto("reset")
            else:
                st.error(msg)

    if st.button("← Back to login"):
        goto("login")

elif st.session_state.page == "reset":
    st.subheader("Enter reset code and new password")
    email = st.session_state.pending_email
    if not email:
        st.info("No pending reset request.")
        if st.button("Go to forgot password"):
            goto("forgot")
    else:
        with st.form("reset_form"):
            code_ = st.text_input("6-digit reset code", max_chars=6)
            new_pw = st.text_input("New password", type="password")
            confirm_pw = st.text_input("Confirm new password", type="password")
            submitted = st.form_submit_button("Reset password", use_container_width=True)

        if submitted:
            ok_pw, pw_msg = is_valid_password(new_pw)
            if new_pw != confirm_pw:
                st.error("Passwords do not match.")
            elif not ok_pw:
                st.error(pw_msg)
            elif not verify_otp(email, code_.strip(), "password_reset"):
                st.error("Invalid or expired code.")
            else:
                update_user_password(email, new_pw)
                st.success("Password reset successfully. Please log in.")
                st.session_state.pending_email = None
                goto("login")

        if st.button("← Back to login"):
            goto("login")


Writing app.py


In [7]:
from db import init_db
init_db()
print("✅ Connected to PostgreSQL and ensured tables exist.")

Database initialized (tables ensured).
✅ Connected to PostgreSQL and ensured tables exist.


In [8]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
time.sleep(1)

# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give the server a moment to boot

public_url = ngrok.connect(8501, "http")
print(f"🚀 Your app is live at: {public_url}")
print("Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.")

🚀 Your app is live at: NgrokTunnel: "https://scorch-snowcap-reprint.ngrok-free.dev" -> "http://localhost:8501"
Open that URL in your browser. Leave this Colab cell/runtime running to keep it up.


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
print("Stopped Streamlit and closed ngrok tunnel.")